In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
mold_list = "../data/WWW Bottom Mold list- All brands - 2026 (2).xlsx"
management_file = "../data/Mold Capacity Management.xlsm"

mold_list_raw_df = pd.read_excel(mold_list, sheet_name="All Brands -done")
vendor_master_raw_df = pd.read_excel(mold_list, sheet_name="Vendor Master")
mold_style_raw_df = pd.read_excel(management_file, sheet_name="Style Mold Info")
mold_vendor_allcation_raw_df = pd.read_excel(management_file, sheet_name="Mold Vendor Allocation")
special_mold_raw_df = pd.read_excel(management_file, sheet_name="Special Molds")

In [ ]:
special_mold_raw_df

In [3]:
def clean_df_col_names(df):
    
    cols = {}
    for col in df.columns:
        new_col = str(col).split("\n")[0].strip()
        cols[col] = new_col.strip().lower().replace(' ', '_').replace('-', '_')
    df.rename(columns=cols, inplace=True)
    return df

def replace_whitespace(df):
    return df.replace(
    to_replace=[
        r"^\s*$",     # empty or whitespace-only
        r"^NA$",      # NA
        r"^N/A$",     # N/A
        r"^\xa0$",    # non-breaking space
    ],
    value=np.nan,
    regex=True
)

def normalize_season(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    # -----------------------------------
    # Normalize season prefixes
    # -----------------------------------
    replacements = {
        "SPR": "S",
        "SS": "S",
        "AW": "F",
        "FW": "F"
    }

    for old, new in replacements.items():
        if value.startswith(old):
            value = value.replace(old, new, 1)

    # -----------------------------------
    # Validate final format
    # -----------------------------------
    pattern = r'^[SF]\d{2}$'

    if re.match(pattern, value):
        return value

    return np.nan

import numpy as np
import pandas as pd


def normalize_location(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().title()

    # -----------------------------------
    # Normalize aliases
    # -----------------------------------
    replacements = {
        "North Vietnam": "Vietnam",
        "South Vietnam": "Vietnam",
        "Viet Nam": "Vietnam",
        "Indonesia ": "Indonesia"
    }

    return replacements.get(value, value)

In [4]:
df =mold_list_raw_df.copy()

df = clean_df_col_names(df) # Clean column names
df = replace_whitespace(df) # replace whitespaces

# Define data types for each column
dtype_set ={
    'brand': str, 
    'part_name': str, 
    'mold_code': str, 
    'compound': str, 
    'vendor_name': str,
    'location': str, 
    'mold_shop': str, 
    'season': str, 
    'a_mold_cost': float, 
    'last_demand_season': str,
    'mold_ownership': str, 
    'mold_condition': str, 
    'daily_output': float, 
    'total_mold_qty': float,
    '1': float, '1.5': float, '2': float, '2.5': float, '3': float, '3.5': float, '4': float, '4.5': float, '5': float, '5.5': float, '6': float, '6.5': float,
    '7': float, '7.5': float, '8': float, '8.5': float, '9': float, '9.5': float, '10': float, '10.5': float, '11': float, '11.5': float, '12': float,
    '17.5': float, '18': float, 
    'comments': str, 'remark': str, 'actual_output': str
}
for col, dtype in dtype_set.items():
    if col in df.columns and dtype == str:
        df[col] = df[col].astype(dtype)
    elif col in df.columns and dtype == float:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        print(f"Warning: Column '{col}' not found in DataFrame.")


# keep only valid season values in 'last_demand_season' & 'season' column
df['season'] = df['season'].apply(normalize_season)
df['last_demand_season'] = df['last_demand_season'].apply(normalize_season)

# split brand
df['brand'] = df['brand'].apply(lambda x: x.split("/"))

# normalize vendor_name
df['vendor_name'] = df['vendor_name'].str.strip().str.upper()

#normalize location
df['location'] = df['location'].apply(normalize_location)

In [5]:
df = df[(df['brand'].apply(lambda x: 'Saucony' in x)) & ((df['part_name'] =="Outsole") | (df['part_name'] =="Midsole"))]

In [6]:
mold_style_df = mold_style_raw_df.iloc[:,:4]

include_mask = (mold_style_df['Outsole_Name'] == mold_style_df['Midsole_Name']) | (mold_style_df['Midsole_Name'].isna()) | (mold_style_df['Outsole_Name'].isna())
mold_style_df = mold_style_df[include_mask]

mold_style_df['Mold_Code'] = mold_style_df.apply(lambda row: row['Outsole_Name'] if pd.notna(row['Outsole_Name']) else row['Midsole_Name'], axis=1)

mold_style_df = mold_style_df[['Brand', 'Style_Name', 'Mold_Code']]
mold_style_df

,Brand,Style_Name,Mold_Code
0,SAUCONY,586I,SA-2562
1,SAUCONY,586I S,SA-2562
2,SAUCONY,AURA TR,SA-2354
3,SAUCONY,AURA TR 2,SA-2555
4,SAUCONY,AURA TR GTX,SA-2354
...,...,...,...
153,SAUCONY,ECHELON 11,SA-2654
154,SAUCONY,EXCURSION TR18,SA-2659
155,SAUCONY,TRIUMPH 24 GTX,SA-2661
156,SAUCONY,SHADOW 5000 (+),SA-6000-NU


In [17]:
mold_vendor_allocation_df = mold_vendor_allcation_raw_df.copy()

mold_vendor_allocation_df.columns = mold_vendor_allocation_df.iloc[0]
mold_vendor_allocation_df = mold_vendor_allocation_df[1:]

mold_vendor_allocation_df = mold_vendor_allocation_df[['Factory Number', 'Mold Code', 'Sole Type', 'Sole Part', 'Vendor ID']]

mold_vendor_allocation_df

,Factory Number,Mold Code,Sole Type,Sole Part,Vendor ID
1,61427,SA-2552,Outsole,OS1 (Top),FTY000575
2,61427,SA-2552,Midsole,MS1 (Top),FTY000575
3,61427,SA-2566,Outsole,OS1 (Top),FTY000575
4,61427,SA-2566,Midsole,MS1 (Top),FTY000575
5,61427,SA-2566,Midsole,MS2 (Bottom),FTY000575
...,...,...,...,...,...
133,61589,SA-2465,Midsole,MS1 (Top),FTY000594
134,62093,SA-2101,Outsole,OS1 (Top),FTY000615
135,62093,SA-2101,Midsole,MS1 (Top),FTY000594
136,61510,SA-2609,Midsole,MS3 (Heel),FTY000720


In [21]:
import pandas as pd
import numpy as np
import re
from datetime import date

# ----------------------------
# Helpers
# ----------------------------
def none_if_nan(x):
    if pd.isna(x):
        return None
    return x

def to_int_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return int(float(x))
    except Exception:
        return x

def to_float_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return x

def normalize_code(s):
    s = none_if_nan(s)
    if s is None:
        return None
    s = str(s).strip().upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or None

def normalize_mold_code(s):
    s = none_if_nan(s)
    if s is None:
        return None
    return str(s).strip().upper()

def normalize_location_code(location_name):
    location_name = none_if_nan(location_name)
    if location_name is None:
        return None

    key = str(location_name).strip().title()
    location_code_map = {
        "Vietnam": "VN",
        "Indonesia": "ID",
        "India": "IN",
        "China": "CN",
        "Cambodia": "KH",
        "Thailand": "TH",
        "Myanmar": "MM",
        "Bangladesh": "BD",
        "Pakistan": "PK",
        "Philippines": "PH",
        "Taiwan": "TW",
        "Korea": "KR",
        "South Korea": "KR",
        "Japan": "JP",
        "Usa": "US",
        "United States": "US",
        "Mexico": "MX"
    }

    if key in location_code_map:
        return location_code_map[key]

    fallback = re.sub(r"[^A-Za-z]", "", key).upper()
    return fallback[:2] if fallback else None

def component_code_from_name(component_name):
    s = none_if_nan(component_name)
    if s is None:
        return None
    s = str(s).strip()
    m = re.match(r"^([^\s(]+)", s)
    return m.group(1) if m else s

def ensure_list(x):
    x = none_if_nan(x)
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [str(x)]

def normalize_col_name(col_name):
    return str(col_name).split("\n")[0].strip().lower().replace(" ", "_").replace("-", "_")

def pick_col(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def normalize_numeric_or_text(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        f = float(x)
        return int(f) if f.is_integer() else f
    except Exception:
        return str(x).strip()

# Size columns: "1", "1.5", ..., "18"
size_cols = [str(x).replace(".0", "") for x in np.arange(1, 18.5, 0.5)]

# ----------------------------
# Build vendor reference from vendor_master_raw_df
# mold_list.vendor_name = vendor_master.vendor_short_name
# ----------------------------
vendor_master_df = vendor_master_raw_df.copy()
vendor_master_df.columns = [normalize_col_name(c) for c in vendor_master_df.columns]
vendor_master_df = replace_whitespace(vendor_master_df)

vm_cols = set(vendor_master_df.columns)
vendor_short_col = pick_col(vm_cols, ["vendor_short_name", "vendor_shortname", "short_name", "vendor_code", "vendor"] )
vendor_id_col = pick_col(vm_cols, ["vendor_id", "vendor_no", "factory_id", "supplier_id", "id"] )
vendor_full_col = pick_col(vm_cols, ["vendor_name", "vendor_full_name", "supplier_name", "factory_name", "name"] )
vendor_note_col = pick_col(vm_cols, ["note", "notes", "remark", "comments"] )

vendor_reference_map = {}
if vendor_short_col is not None:
    for _, vm_row in vendor_master_df.iterrows():
        vendor_short_name = none_if_nan(vm_row.get(vendor_short_col))
        if vendor_short_name is None:
            continue

        vendor_code = normalize_code(vendor_short_name)
        if vendor_code is None:
            continue

        vendor_full_name = none_if_nan(vm_row.get(vendor_full_col)) if vendor_full_col else None
        vendor_id_value = none_if_nan(vm_row.get(vendor_id_col)) if vendor_id_col else None
        vendor_note_value = none_if_nan(vm_row.get(vendor_note_col)) if vendor_note_col else None

        vendor_reference_map[vendor_code] = {
            "name": vendor_full_name if vendor_full_name else vendor_short_name,
            "shortName": vendor_short_name,
            "vendorId": vendor_id_value,
            "notes": vendor_note_value
        }

# ----------------------------
# Build lookup from mold_style_df
# ----------------------------
style_map_by_mold = {}
for _, style_row in mold_style_df.iterrows():
    mold_code_key = normalize_mold_code(style_row.get("Mold_Code"))
    if mold_code_key is None:
        continue

    style_item = {
        "brand": none_if_nan(style_row.get("Brand")),
        "styleName": none_if_nan(style_row.get("Style_Name"))
    }

    style_list = style_map_by_mold.setdefault(mold_code_key, [])
    if style_item not in style_list:
        style_list.append(style_item)

# ----------------------------
# Build lookup from mold_vendor_allocation_df
# ----------------------------
allocation_map_by_mold = {}
for _, alloc_row in mold_vendor_allocation_df.iterrows():
    mold_code_key = normalize_mold_code(alloc_row.get("Mold Code"))
    if mold_code_key is None:
        continue

    alloc_item = {
        "factoryNumber": none_if_nan(alloc_row.get("Factory Number")),
        "soleType": none_if_nan(alloc_row.get("Sole Type")),
        "solePart": none_if_nan(alloc_row.get("Sole Part")),
        "vendorId": none_if_nan(alloc_row.get("Vendor ID"))
    }

    alloc_list = allocation_map_by_mold.setdefault(mold_code_key, [])
    if alloc_item not in alloc_list:
        alloc_list.append(alloc_item)

# ----------------------------
# Build lookup from special_mold_raw_df for sizing rules
# key: (mold_code, component_code) -> {mold_size: [shoe_sizes...] }
# ----------------------------
special_sizing_by_mold_component = {}
special_df = special_mold_raw_df.copy()
special_df = replace_whitespace(special_df)

special_size_cols = [c for c in special_df.columns if str(c).strip().startswith("Size ")]
for _, sp_row in special_df.iterrows():
    mold_code_key = normalize_mold_code(sp_row.get("Mold Code"))
    component_code_key = component_code_from_name(sp_row.get("Sole Part"))

    if mold_code_key is None or component_code_key is None:
        continue

    comp_key = str(component_code_key)
    key = (mold_code_key, comp_key)

    mold_to_shoes = special_sizing_by_mold_component.setdefault(key, {})
    for col in special_size_cols:
        mold_size = str(col).replace("Size", "").strip()
        shoe_size = normalize_numeric_or_text(sp_row.get(col))
        if shoe_size is None:
            continue

        shoe_list = mold_to_shoes.setdefault(mold_size, [])
        if shoe_size not in shoe_list:
            shoe_list.append(shoe_size)

# ----------------------------
# v2 Container
# ----------------------------
result = {
    "schemaVersion": "2.0",
    "lastUpdated": date.today().isoformat(),
    "reference": {
        "vendors": vendor_reference_map,
        "moldShops": {},
        "locations": {},
        "allowedSoleTypes": ["Outsole", "Midsole"]
    },
    "families": {}
}

# ----------------------------
# Build JSON
# ----------------------------
for _, row in df.iterrows():

    mold_code = none_if_nan(row.get("mold_code")) or none_if_nan(row.get("mold_code_note"))
    if mold_code is None:
        continue
    mold_code_key = normalize_mold_code(mold_code)

    brand = row.get("brand")
    sole_type = none_if_nan(row.get("part_name"))
    component_name = none_if_nan(row.get("component_name"))
    compound = none_if_nan(row.get("compound"))

    component_code = component_code_from_name(component_name)
    if component_code is None:
        continue

    vendor_name = none_if_nan(row.get("vendor_name"))
    mold_shop_name = none_if_nan(row.get("mold_shop"))
    location = none_if_nan(row.get("location"))

    vendor_code = normalize_code(vendor_name) if vendor_name else None
    mold_shop_code = normalize_code(mold_shop_name) if mold_shop_name else None
    location_code = normalize_location_code(location)

    # ----------------------------
    # Reference catalogs
    # ----------------------------
    if vendor_code and vendor_code not in result["reference"]["vendors"]:
        # Fallback when vendor does not exist in Vendor Master
        result["reference"]["vendors"][vendor_code] = {
            "name": vendor_name,
            "shortName": vendor_name,
            "vendorId": None,
            "notes": none_if_nan(row.get("vendor_name_note"))
        }

    if mold_shop_code and mold_shop_code not in result["reference"]["moldShops"]:
        result["reference"]["moldShops"][mold_shop_code] = {"name": mold_shop_name}

    if location_code and location_code not in result["reference"]["locations"]:
        result["reference"]["locations"][location_code] = {"name": location}

    # ----------------------------
    # Family block
    # ----------------------------
    fam = result["families"].setdefault(str(mold_code), {
        "moldCode": str(mold_code),
        "brands": [],
        "notes": None,
        "stylesUsingThisFamily": [],
        "sourcingRules": [],
        "components": {},

        # ### NEW: sizing rules live at family level (easier to export 1 sheet per family)
        "sizingRulesByComponent": {}
    })

    # fill from mold_style_df
    if mold_code_key in style_map_by_mold and not fam["stylesUsingThisFamily"]:
        fam["stylesUsingThisFamily"] = style_map_by_mold[mold_code_key]

    # fill from mold_vendor_allocation_df
    if mold_code_key in allocation_map_by_mold and not fam["sourcingRules"]:
        fam["sourcingRules"] = allocation_map_by_mold[mold_code_key]

    # ----------------------------
    # Component block
    # ----------------------------
    if sole_type is None:
        sole_type = "Unknown"

    sole_block = fam["components"].setdefault(str(sole_type), {})

    comp = sole_block.setdefault(str(component_code), {
        "displayName": component_name,
        "compound": compound,
        "notes": None,
        "molds": []
    })

    if comp.get("compound") is None and compound is not None:
        comp["compound"] = compound

    # ----------------------------
    # Inventory qtyByMoldSize
    # ----------------------------
    qty_by_mold_size = {}
    for s in size_cols:
        val = row.get(s)
        if pd.notna(val):
            qty_by_mold_size[s] = to_int_or_none(val)

    # ensure sizing keys exist with NULL values by default
    sizing_comp = fam["sizingRulesByComponent"].setdefault(str(component_code), {
        "componentName": comp["displayName"],
        "soleType": str(sole_type),
        "moldSizeToShoeSizes": {},
        "notes": "Auto-generated keys from inventory; values pending"
    })

    for mold_size in qty_by_mold_size.keys():
        # only create if missing; DO NOT overwrite if already filled later
        sizing_comp["moldSizeToShoeSizes"].setdefault(str(mold_size), None)

    # overlay special mold sizing map when available
    sp_key = (mold_code_key, str(component_code))
    sp_mold_to_shoes = special_sizing_by_mold_component.get(sp_key, {})
    if sp_mold_to_shoes:
        for mold_size, shoe_sizes in sp_mold_to_shoes.items():
            sizing_comp["moldSizeToShoeSizes"][str(mold_size)] = shoe_sizes
        sizing_comp["notes"] = "Merged from Special Molds sheet and inventory defaults"

    # ----------------------------
    # Mold record
    # ----------------------------
    vendor_reference = result["reference"]["vendors"].get(vendor_code, {}) if vendor_code else {}
    mold_record = {
        "vendorCode": vendor_code,
        "vendorName": vendor_reference.get("name") if vendor_reference.get("name") else vendor_name,
        "vendorShortName": vendor_reference.get("shortName") if vendor_reference.get("shortName") else vendor_name,
        "vendorId": vendor_reference.get("vendorId"),
        "moldShopCode": mold_shop_code,
        "moldShopName": mold_shop_name,
        "location": {
            "code": location_code,
            "name": location
        },
        "initSeason": none_if_nan(row.get("season")),
        "lastDemandSeason": none_if_nan(row.get("last_demand_season")),
        "capacity": {
            "dailyOutput": to_float_or_none(row.get("daily_output")),
            "actualOutput": to_float_or_none(row.get("actual_output"))
        },
        "asset": {
            "moldCost": to_float_or_none(row.get("a_mold_cost")),
            "ownership": none_if_nan(row.get("mold_ownership")),
            "condition": none_if_nan(row.get("mold_condition")),
            "conditionNote": none_if_nan(row.get("mold_condition_note"))
        },
        "inventory": {
            "totalMoldQty": to_int_or_none(row.get("total_mold_qty")),
            "qtyByMoldSize": qty_by_mold_size
        },
        "comments": none_if_nan(row.get("comments")),
        "remark": none_if_nan(row.get("remark"))
    }

    comp["molds"].append(mold_record)

In [ ]:
# Fix direction for Special Molds mapping:
# sheet columns are Shoe Size, cell values are Mold Size
# convert to JSON shape: moldSizeToShoeSizes[mold_size] = [shoe_sizes]

def _fmt_size(x):
    x = normalize_numeric_or_text(x)
    if x is None:
        return None
    if isinstance(x, (int, float)):
        xf = float(x)
        return str(int(xf)) if xf.is_integer() else str(xf)
    return str(x).strip()

def _is_numeric_size(value):
    if value is None:
        return False
    try:
        float(str(value).strip())
        return True
    except Exception:
        return False

special_df = replace_whitespace(special_mold_raw_df.copy())
special_size_cols = [c for c in special_df.columns if str(c).strip().startswith("Size ")]

for _, sp_row in special_df.iterrows():
    mold_code_key = normalize_mold_code(sp_row.get("Mold Code"))
    component_code_key = component_code_from_name(sp_row.get("Sole Part"))

    if mold_code_key is None or component_code_key is None:
        continue

    fam = result.get("families", {}).get(str(mold_code_key))
    if not fam:
        continue

    comp_code = str(component_code_key)
    sizing_comp = fam.get("sizingRulesByComponent", {}).get(comp_code)
    if sizing_comp is None:
        continue

    corrected_map = {}
    for col in special_size_cols:
        shoe_size = _fmt_size(str(col).replace("Size", "").strip())
        mold_size = _fmt_size(sp_row.get(col))

        if mold_size is None or shoe_size is None:
            continue
        if not _is_numeric_size(mold_size):
            continue

        corrected_map.setdefault(mold_size, [])
        if shoe_size not in corrected_map[mold_size]:
            corrected_map[mold_size].append(shoe_size)

    if not corrected_map:
        continue

    old_map = sizing_comp.get("moldSizeToShoeSizes", {}) or {}
    new_map = {}

    # keep existing default placeholders only, and only numeric mold-size keys
    for k, v in old_map.items():
        if v is None and _is_numeric_size(k):
            new_map[str(k)] = None

    # apply corrected non-null mapping from Special Molds
    for mold_size, shoe_sizes in corrected_map.items():
        new_map[str(mold_size)] = shoe_sizes

    sizing_comp["moldSizeToShoeSizes"] = new_map
    sizing_comp["notes"] = "Special Molds mapping applied (Mold Size -> Shoe Sizes)"

In [23]:
import json

with open("../data/mold_data.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False, default=str)